# RIS OGD API v2.6 — Court Decision Scraper

**Purpose:** Query Austrian court decisions (Judikatur) via the official RIS Open Government Data API for keywords related to parental alienation and child welfare.

**API Base:** `https://data.bka.gv.at/ris/api/v2.6/`  
**Documentation:** [API Handbook (PDF)](https://data.bka.gv.at/ris/ogd/v2.6/Documents/Dokumentation_OGD-RIS_API.pdf)  
**Data catalog:** [data.gv.at](https://www.data.gv.at/katalog/dataset/ris2_6)  

**Keywords:** `Entfremdung`, `Kindeswohl`, `elterliche Entfremdung`, `Kontaktrecht`, `Obsorge`

---

### Why the API instead of HTML scraping?
- Official government OGD endpoint — no legal/ethical concerns
- Returns structured JSON — no fragile CSS selectors
- Stable and versioned (v2.6)
- No API key required
- Includes full decision text in the response

### How the API works
The RIS API uses simple GET requests with URL parameters:
```
https://data.bka.gv.at/ris/api/v2.6/Judikatur?Applikation=Justiz&Suchworte=Kindeswohl
```

Key parameters for Judikatur/Justiz:
- `Applikation` — which court: `Justiz` (OGH/OLG/LG/BG), `Vfgh`, `Vwgh`, `Bvwg`, `Lvwg`
- `Suchworte` — keyword search across full text
- `Geschaeftszahl` — case number search
- `EntscheidungsdatumVon` / `EntscheidungsdatumBis` — date range (YYYY-MM-DD)
- `Norm` — search by referenced legal norm
- `Seite` — page number (1-based)
- `Seitengroesse` — results per page (default 20, max 100)
- `DokumenteProSeite` — full-text inclusion flag

## 0. Setup

In [1]:
import requests
import pandas as pd
import time
import re
import json
import os
import logging
from datetime import datetime
from pathlib import Path
from urllib.parse import urlencode, quote

# --- Configuration ---
API_BASE = "https://data.bka.gv.at/ris/api/v2.6"

KEYWORDS = [
    "Entfremdung",
    "Kindeswohl",
    "elterliche Entfremdung",
    "Kontaktrecht",
    "Obsorge",
]

# Which court applications to query
# "Justiz" = OGH, OLG, LG, BG (regular courts — most relevant for family law)
# "Vfgh" = Constitutional Court, "Vwgh" = Administrative Court
APPLICATIONS = ["Justiz"]

# Page size (max 100 per the API docs)
PAGE_SIZE = 100

# Polite delay between requests (seconds)
REQUEST_DELAY = 1.0

# Output directory
OUTPUT_DIR = Path("ris_data")
OUTPUT_DIR.mkdir(exist_ok=True)

# Logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s — %(levelname)s — %(message)s")
logger = logging.getLogger(__name__)

print(f"API Base: {API_BASE}")
print(f"Output directory: {OUTPUT_DIR.resolve()}")
print(f"Keywords: {KEYWORDS}")
print(f"Applications: {APPLICATIONS}")

/Users/maksimsmirnov/Desktop/thesis/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


API Base: https://data.bka.gv.at/ris/api/v2.6
Output directory: /Users/maksimsmirnov/Desktop/thesis/ris_data
Keywords: ['Entfremdung', 'Kindeswohl', 'elterliche Entfremdung', 'Kontaktrecht', 'Obsorge']
Applications: ['Justiz']


In [2]:
# --- HTTP Session ---
session = requests.Session()
session.headers.update({
    "Accept": "application/json",
    "User-Agent": "MasterThesis-RIS-Research/1.0 (Academic Research)",
})


def api_get(endpoint, params, retries=3):
    """
    Make a GET request to the RIS API with retry logic.
    Returns parsed JSON or None on failure.
    """
    url = f"{API_BASE}/{endpoint}"
    
    for attempt in range(retries):
        try:
            time.sleep(REQUEST_DELAY)
            resp = session.get(url, params=params, timeout=30)
            resp.raise_for_status()
            return resp.json()
        except requests.exceptions.HTTPError as e:
            logger.warning(f"HTTP {resp.status_code} on attempt {attempt+1}: {e}")
            if resp.status_code == 429:  # rate limited
                time.sleep(10 * (attempt + 1))
            elif resp.status_code >= 500:
                time.sleep(5 * (attempt + 1))
            else:
                return None  # client error, don't retry
        except requests.exceptions.RequestException as e:
            logger.warning(f"Request error on attempt {attempt+1}: {e}")
            time.sleep(5 * (attempt + 1))
        except ValueError as e:
            logger.warning(f"JSON parse error: {e}")
            # The API might return XML instead of JSON — let's handle that
            logger.info(f"Raw response: {resp.text[:500]}")
            return None
    
    logger.error(f"All retries failed for {url}")
    return None


print("Session ready.")

Session ready.


## 1. Explore the API — Test Request

Let's make a single test query to understand the response structure.

In [3]:
# --- Test: search Justiz for "Kindeswohl" ---
test_params = {
    "Applikation": "Justiz",
    "Suchworte": "Kindeswohl",
    "Seite": 1,
    "Seitengroesse": 5,  # just 5 results for testing
}

test_url = f"{API_BASE}/Judikatur?{urlencode(test_params)}"
print(f"Test URL: {test_url}\n")

test_data = api_get("Judikatur", test_params)

if test_data:
    print("✅ API returned data successfully!\n")
    
    # Show top-level structure
    print("Top-level keys:", list(test_data.keys()))
    print()
    
    # Pretty-print the response (truncated)
    print(json.dumps(test_data, indent=2, ensure_ascii=False)[:3000])
else:
    print("❌ API request failed. Check the URL and parameters.")
    print(f"Try opening this in your browser: {test_url}")

Test URL: https://data.bka.gv.at/ris/api/v2.6/Judikatur?Applikation=Justiz&Suchworte=Kindeswohl&Seite=1&Seitengroesse=5

✅ API returned data successfully!

Top-level keys: ['OgdSearchResult']

{
  "OgdSearchResult": {
    "OgdDocumentResults": {
      "Hits": {
        "@pageNumber": "1",
        "@pageSize": "20",
        "#text": "189"
      },
      "OgdDocumentReference": [
        {
          "Data": {
            "Metadaten": {
              "Technisch": {
                "ID": "JJR_20130508_OGH0002_0060OB00041_13T0000_004",
                "Applikation": "Justiz",
                "Organ": "OGH",
                "ImportTimestamp": {
                  "@xsi:nil": "true",
                  "@xmlns:xsi": "http://www.w3.org/2001/XMLSchema-instance"
                }
              },
              "Allgemein": {
                "Veroeffentlicht": "2013-07-09",
                "Geaendert": "2026-02-24",
                "DokumentUrl": "https://www.ris.bka.gv.at/Dokument.wxe?Abfrage=Just

In [4]:
# --- 1.1 Understand the response structure ---
# This cell helps you map out what fields are available.

if test_data:
    # The response typically has: OgdSearchResult -> OgdDocumentResults -> OgdDocumentReference[]
    # Let's walk through it
    
    # Try common response structures
    # Structure might be nested differently — adapt based on test output above
    
    def explore_keys(d, prefix="", depth=0):
        """Recursively print keys of a nested dict/list."""
        if depth > 4:
            return
        if isinstance(d, dict):
            for key, val in d.items():
                val_type = type(val).__name__
                if isinstance(val, list):
                    val_type = f"list[{len(val)}]"
                elif isinstance(val, str):
                    val_type = f"str({len(val)} chars)"
                print(f"{'  ' * depth}{prefix}{key}: {val_type}")
                if isinstance(val, (dict, list)):
                    explore_keys(val, "", depth + 1)
        elif isinstance(d, list) and d:
            print(f"{'  ' * depth}[0]:")
            explore_keys(d[0], "", depth + 1)
    
    print("Response structure:")
    print("=" * 50)
    explore_keys(test_data)

Response structure:
OgdSearchResult: dict
  OgdDocumentResults: dict
    Hits: dict
      @pageNumber: str(1 chars)
      @pageSize: str(2 chars)
      #text: str(3 chars)
    OgdDocumentReference: list[20]
      [0]:
        Data: dict


In [5]:
# --- 1.2 Extract a single document to see all fields ---

if test_data:
    # Navigate to documents — adjust path based on 1.1 output
    # Common paths:
    #   test_data["OgdSearchResult"]["OgdDocumentResults"]["OgdDocumentReference"]
    #   test_data["results"]
    #   test_data["Hits"]
    
    # Try multiple possible structures
    documents = None
    total_hits = None
    
    # Path A: OGD-style nested
    if "OgdSearchResult" in test_data:
        sr = test_data["OgdSearchResult"]
        if "OgdDocumentResults" in sr:
            dr = sr["OgdDocumentResults"]
            if "OgdDocumentReference" in dr:
                documents = dr["OgdDocumentReference"]
                if not isinstance(documents, list):
                    documents = [documents]  # single result
        # Total hits
        if "Hits" in sr:
            total_hits = sr.get("Hits", {}).get("#text", sr.get("Hits"))
    
    # Path B: flat
    if documents is None:
        for key in ["results", "documents", "Documents", "Hits"]:
            if key in test_data and isinstance(test_data[key], list):
                documents = test_data[key]
                break
    
    if documents:
        print(f"Found {len(documents)} documents (total hits: {total_hits})\n")
        print("First document keys:")
        print(list(documents[0].keys()))
        print("\nFirst document (full):")
        # Print but truncate long text fields
        doc = documents[0].copy()
        for k, v in doc.items():
            if isinstance(v, str) and len(v) > 500:
                doc[k] = v[:500] + f"... [{len(v)} chars total]"
        print(json.dumps(doc, indent=2, ensure_ascii=False))
    else:
        print("Could not locate documents in response. Check structure in cell 1.1.")
        print("You may need to adapt the extraction logic below.")

Found 20 documents (total hits: None)

First document keys:
['Data']

First document (full):
{
  "Data": {
    "Metadaten": {
      "Technisch": {
        "ID": "JJR_20130508_OGH0002_0060OB00041_13T0000_004",
        "Applikation": "Justiz",
        "Organ": "OGH",
        "ImportTimestamp": {
          "@xsi:nil": "true",
          "@xmlns:xsi": "http://www.w3.org/2001/XMLSchema-instance"
        }
      },
      "Allgemein": {
        "Veroeffentlicht": "2013-07-09",
        "Geaendert": "2026-02-24",
        "DokumentUrl": "https://www.ris.bka.gv.at/Dokument.wxe?Abfrage=Justiz&Dokumentnummer=JJR_20130508_OGH0002_0060OB00041_13T0000_004"
      },
      "Judikatur": {
        "Dokumenttyp": "Rechtssatz",
        "Geschaeftszahl": {
          "item": "6Ob41/13t; 4Ob32/13d; 6Ob74/13w; 4Ob58/13b; 1Ob126/13f; 6Ob155/13g; 3Ob145/13i; 3Ob103/13p; 7Ob211/13z; 5Ob227/13p; 1Ob220/13d; 10Ob53/13m; 7Ob64/14h; 4Ob88/14s; 5Ob144/14h; 3Ob128/14s; 1Ob156/14v; 3Ob149/14d; 7Ob198/14i; 1Ob250/14t; 2Ob2

## 2. Response Parser

Based on the exploration above, define functions to extract documents and metadata.

**⚠️ You may need to adapt field names** based on what you see in Section 1.

In [6]:
def parse_api_response(data):
    """
    Parse the RIS API JSON response.
    Returns (total_hits: int, documents: list[dict]).
    
    ADAPT FIELD NAMES HERE based on the test output in Section 1.
    """
    if not data:
        return 0, []
    
    documents = []
    total_hits = 0
    
    # --- Navigate to the document list ---
    # Structure A: OGD nested format (most likely)
    try:
        sr = data.get("OgdSearchResult", data)
        
        # Extract total hits
        hits_raw = sr.get("Hits", {})
        if isinstance(hits_raw, dict):
            total_hits = int(hits_raw.get("#text", hits_raw.get("value", 0)))
        elif isinstance(hits_raw, (int, str)):
            total_hits = int(hits_raw)
        
        # Extract documents
        dr = sr.get("OgdDocumentResults", {})
        docs_raw = dr.get("OgdDocumentReference", [])
        
        if isinstance(docs_raw, dict):
            docs_raw = [docs_raw]  # single result
        
        documents = docs_raw if isinstance(docs_raw, list) else []
        
    except (KeyError, TypeError, AttributeError) as e:
        logger.warning(f"Error parsing response: {e}")
        # Try flat structure
        for key in ["results", "documents", "Documents"]:
            if key in data and isinstance(data[key], list):
                documents = data[key]
                break
    
    return total_hits, documents


def extract_document_fields(doc):
    """
    Extract relevant fields from a single document reference.
    Returns a flat dict suitable for a DataFrame row.
    
    ADAPT FIELD NAMES based on the test output.
    Common RIS API field names for Justiz:
      - Geschaeftszahl, Gericht, Entscheidungsdatum, Norm
      - Dokumentnummer, DokumentUrl
      - Kurzinformation (short info/summary)
      - Data.Dokumentliste.ContentReference.Urls.ContentUrl (full text URL)
    """
    if not doc:
        return {}
    
    # Helper: safely get nested values
    def safe_get(*keys, default=""):
        val = doc
        for k in keys:
            if isinstance(val, dict):
                val = val.get(k, default)
            else:
                return default
        return val if val is not None else default
    
    # --- Extract standard fields ---
    result = {
        "dokumentnummer": safe_get("Dokumentnummer"),
        "geschaeftszahl": safe_get("Geschaeftszahl"),
        "gericht": safe_get("Gericht"),
        "entscheidungsdatum": safe_get("Entscheidungsdatum"),
        "norm": safe_get("Norm"),
        "kurzinformation": safe_get("Kurzinformation"),
        "dokument_url": safe_get("DokumentUrl"),
        # V2.6 additions for Justiz
        "rechtsgebiet": safe_get("Rechtsgebiet", default=""),
        "fachgebiet": safe_get("Fachgebiet", default=""),
        "entscheidungsart": safe_get("Entscheidungsart", default=""),
        "spruch": safe_get("Spruch", default=""),
    }
    
    # --- Extract full text ---
    # The full text might be in nested Content/Nutzdaten structure
    # or it might require a separate getDocument call.
    # Let's try to find it in the response first:
    full_text = ""
    
    # Path 1: Direct text content
    data_section = safe_get("Data", default={})
    if isinstance(data_section, dict):
        # Look for Dokumentliste -> ContentReference -> Urls
        docliste = data_section.get("Dokumentliste", {})
        if isinstance(docliste, dict):
            content_ref = docliste.get("ContentReference", {})
            if isinstance(content_ref, list):
                content_ref = content_ref[0] if content_ref else {}
            urls = content_ref.get("Urls", {})
            if isinstance(urls, dict):
                content_urls = urls.get("ContentUrl", [])
                if isinstance(content_urls, dict):
                    content_urls = [content_urls]
                for cu in (content_urls if isinstance(content_urls, list) else []):
                    if isinstance(cu, dict):
                        result["content_url"] = cu.get("Url", cu.get("url", ""))
    
    # Path 2: Nutzdaten (payload data) might contain the actual text
    nutzdaten = safe_get("Nutzdaten", default="")
    if isinstance(nutzdaten, str) and len(nutzdaten) > 100:
        full_text = nutzdaten
    
    # Path 3: Entscheidungstexte might be present
    entscheidungstext = safe_get("Entscheidungstext", safe_get("Entscheidungstexte", default=""))
    if isinstance(entscheidungstext, str) and len(entscheidungstext) > len(full_text):
        full_text = entscheidungstext
    
    result["full_text"] = full_text
    result["has_full_text"] = len(full_text) > 100
    
    return result


# --- Quick test ---
if test_data:
    total, docs = parse_api_response(test_data)
    print(f"Parsed: {total} total hits, {len(docs)} documents on this page")
    if docs:
        parsed = extract_document_fields(docs[0])
        for k, v in parsed.items():
            display_val = str(v)[:200] if isinstance(v, str) else v
            print(f"  {k}: {display_val}")

Parsed: 0 total hits, 20 documents on this page
  dokumentnummer: 
  geschaeftszahl: 
  gericht: 
  entscheidungsdatum: 
  norm: 
  kurzinformation: 
  dokument_url: 
  rechtsgebiet: 
  fachgebiet: 
  entscheidungsart: 
  spruch: 
  content_url: https://www.ris.bka.gv.at/Dokumente/Justiz/JJR_20130508_OGH0002_0060OB00041_13T0000_004/JJR_20130508_OGH0002_0060OB00041_13T0000_004.pdf
  full_text: 
  has_full_text: False


## 3. Fetch Full Texts (if not in search results)

The search endpoint may or may not include full decision text. If it doesn't, we need to fetch each document individually.

There are two options:
1. **SOAP Service** at `data.bka.gv.at/ris/ogd/v2.6/ogdrisservice.asmx` — has a `getDocument` method
2. **Fetch the DokumentUrl** returned in search results — links to the RIS website

Let's first check if the search results include text. If they do, we can skip this section.

In [ ]:
# --- 3.0 Check if search results include full text ---

if test_data:
    _, docs = parse_api_response(test_data)
    if docs:
        sample = extract_document_fields(docs[0])
        if sample.get("has_full_text"):
            print("✅ Full text IS included in search results!")
            print(f"   Text length: {len(sample['full_text'])} chars")
            print("   → You can skip Section 3 and go directly to Section 4.")
        else:
            print("⚠️ Full text NOT in search results.")
            print("   → Need to fetch individually. Continue with Section 3.")
            if sample.get("content_url"):
                print(f"   Content URL found: {sample['content_url'][:100]}")
            if sample.get("dokument_url"):
                print(f"   Dokument URL: {sample['dokument_url'][:100]}")

In [ ]:
# --- 3.1 Fetch individual document via SOAP service ---
# Only needed if the REST API search doesn't include full text.

from xml.etree import ElementTree as ET

SOAP_URL = "https://data.bka.gv.at/ris/ogd/v2.6/OGDRisService.asmx"


def fetch_document_soap(dokumentnummer, applikation="Justiz"):
    """
    Fetch a single document's full content via the SOAP getDocument method.
    Returns the full text as a string.
    """
    soap_body = f"""<?xml version="1.0" encoding="utf-8"?>
<soap:Envelope xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance"
               xmlns:xsd="http://www.w3.org/2001/XMLSchema"
               xmlns:soap="http://schemas.xmlsoap.org/soap/envelope/">
  <soap:Body>
    <getDocument xmlns="http://ogd.bka.gv.at/">
      <application>{applikation}</application>
      <documentNumber>{dokumentnummer}</documentNumber>
    </getDocument>
  </soap:Body>
</soap:Envelope>"""
    
    headers = {
        "Content-Type": "text/xml; charset=utf-8",
        "SOAPAction": "http://ogd.bka.gv.at/getDocument",
    }
    
    try:
        time.sleep(REQUEST_DELAY)
        resp = requests.post(SOAP_URL, data=soap_body.encode("utf-8"), headers=headers, timeout=30)
        resp.raise_for_status()
        
        # The SOAP response wraps the document XML inside a getDocumentResult element
        # We need to parse the outer SOAP envelope, then the inner XML
        root = ET.fromstring(resp.text)
        
        # Find all text content — the structure is complex, so let's just grab all text
        all_text = []
        for elem in root.iter():
            if elem.text and elem.text.strip():
                all_text.append(elem.text.strip())
        
        full_text = "\n".join(all_text)
        return full_text
    
    except Exception as e:
        logger.warning(f"SOAP fetch failed for {dokumentnummer}: {e}")
        return ""


# --- Alternative: fetch the HTML document page and extract text ---
from bs4 import BeautifulSoup


def fetch_document_html(dokument_url):
    """
    Fallback: fetch full text from the RIS website HTML page.
    Use if both REST and SOAP don't include the text.
    """
    if not dokument_url:
        return ""
    
    try:
        time.sleep(REQUEST_DELAY)
        resp = session.get(dokument_url, timeout=30)
        resp.raise_for_status()
        resp.encoding = "utf-8"
        
        soup = BeautifulSoup(resp.text, "lxml")
        
        # Try to find the main content area
        main = (
            soup.select_one(".documentContent, .contentBlock, #DocumentText, .doc-content")
            or soup.find("main")
            or soup.find("body")
        )
        
        if main:
            return main.get_text("\n", strip=True)
        return ""
    
    except Exception as e:
        logger.warning(f"HTML fetch failed for {dokument_url[:80]}: {e}")
        return ""


print("Document fetchers ready.")
print("- fetch_document_soap(): via SOAP service")
print("- fetch_document_html(): fallback via HTML scraping")

In [ ]:
# --- 3.2 Test document fetch ---

if test_data:
    _, docs = parse_api_response(test_data)
    if docs:
        sample = extract_document_fields(docs[0])
        
        # Test SOAP
        if sample.get("dokumentnummer"):
            print(f"Testing SOAP fetch for: {sample['dokumentnummer']}")
            soap_text = fetch_document_soap(sample["dokumentnummer"])
            print(f"SOAP text length: {len(soap_text)} chars")
            if soap_text:
                print(f"Preview: {soap_text[:500]}")
        
        print("\n" + "="*50)
        
        # Test HTML fallback
        if sample.get("dokument_url"):
            print(f"\nTesting HTML fetch for: {sample['dokument_url'][:80]}")
            html_text = fetch_document_html(sample["dokument_url"])
            print(f"HTML text length: {len(html_text)} chars")
            if html_text:
                print(f"Preview: {html_text[:500]}")

## 4. Main Collection Loop

Iterate over all keywords, paginate through results, and collect data.

In [ ]:
# --- Configuration ---

# Max results per keyword (set to None for all)
MAX_RESULTS_PER_KEYWORD = 500

# Date range (optional, format: YYYY-MM-DD)
DATE_FROM = None  # e.g., "2000-01-01"
DATE_TO = None

# Whether to fetch full text for each document (slow but thorough)
# Set to False if text is already included in search results
FETCH_FULL_TEXT = True  # will be auto-adjusted after first test

# Preferred method for fetching full text: "soap", "html", or "none"
FETCH_METHOD = "soap"  # try soap first, fall back to html

print(f"Max results/keyword: {MAX_RESULTS_PER_KEYWORD}")
print(f"Date range: {DATE_FROM or 'any'} to {DATE_TO or 'any'}")
print(f"Fetch full text: {FETCH_FULL_TEXT} (method: {FETCH_METHOD})")

In [ ]:
def search_ris(keyword, applikation="Justiz", max_results=None,
               date_from=None, date_to=None):
    """
    Search RIS Judikatur API for a keyword with pagination.
    Returns list of parsed document dicts.
    """
    all_docs = []
    page = 1
    seen_ids = set()
    
    while True:
        params = {
            "Applikation": applikation,
            "Suchworte": keyword,
            "Seite": page,
            "Seitengroesse": PAGE_SIZE,
        }
        
        if date_from:
            params["EntscheidungsdatumVon"] = date_from
        if date_to:
            params["EntscheidungsdatumBis"] = date_to
        
        data = api_get("Judikatur", params)
        total_hits, docs = parse_api_response(data)
        
        if page == 1:
            effective_max = min(total_hits, max_results) if max_results else total_hits
            logger.info(f"'{keyword}' ({applikation}): {total_hits} hits, collecting up to {effective_max}")
        
        if not docs:
            logger.info(f"  No more results at page {page}")
            break
        
        # Parse and deduplicate
        new_count = 0
        for doc in docs:
            parsed = extract_document_fields(doc)
            doc_id = parsed.get("dokumentnummer") or parsed.get("geschaeftszahl") or str(len(all_docs))
            
            if doc_id not in seen_ids:
                seen_ids.add(doc_id)
                parsed["search_keyword"] = keyword
                parsed["search_applikation"] = applikation
                all_docs.append(parsed)
                new_count += 1
        
        logger.info(f"  Page {page}: +{new_count} new docs (total: {len(all_docs)})")
        
        # Check limits
        if max_results and len(all_docs) >= max_results:
            all_docs = all_docs[:max_results]
            break
        
        if len(docs) < PAGE_SIZE:
            break  # last page
        
        page += 1
    
    return all_docs


print("Search function ready.")

In [ ]:
# --- 4.1 Collect all search results ---

all_results = []

for applikation in APPLICATIONS:
    for kw in KEYWORDS:
        logger.info(f"\n{'='*60}")
        logger.info(f"Searching: {applikation} / {kw}")
        logger.info(f"{'='*60}")
        
        results = search_ris(
            keyword=kw,
            applikation=applikation,
            max_results=MAX_RESULTS_PER_KEYWORD,
            date_from=DATE_FROM,
            date_to=DATE_TO,
        )
        all_results.extend(results)

# --- Deduplicate across keywords ---
seen = {}
for r in all_results:
    key = r.get("dokumentnummer") or r.get("geschaeftszahl") or r.get("dokument_url")
    if key in seen:
        existing_kw = seen[key]["search_keyword"]
        if r["search_keyword"] not in existing_kw:
            seen[key]["search_keyword"] = f"{existing_kw}; {r['search_keyword']}"
    else:
        seen[key] = r

unique_results = list(seen.values())
logger.info(f"\nTotal unique decisions: {len(unique_results)}")

# Save intermediate metadata
df_meta = pd.DataFrame(unique_results)
df_meta.to_csv(OUTPUT_DIR / "search_results_meta.csv", index=False, encoding="utf-8-sig")
print(f"Saved metadata: {OUTPUT_DIR / 'search_results_meta.csv'}")

In [ ]:
# --- 4.2 Fetch full texts (if needed) ---

CHECKPOINT_FILE = OUTPUT_DIR / "decisions_checkpoint.json"

# Check if we actually need to fetch full texts
texts_already_present = sum(1 for r in unique_results if r.get("has_full_text"))
print(f"Results with full text from search: {texts_already_present} / {len(unique_results)}")

if texts_already_present == len(unique_results):
    print("\n✅ All results already include full text! Skipping individual fetches.")
    collected = unique_results
else:
    print(f"\nNeed to fetch {len(unique_results) - texts_already_present} full texts...")
    
    # Load checkpoint
    if CHECKPOINT_FILE.exists():
        with open(CHECKPOINT_FILE, "r", encoding="utf-8") as f:
            collected = json.load(f)
        fetched_ids = {d.get("dokumentnummer") for d in collected}
        logger.info(f"Resuming: {len(collected)} already fetched")
    else:
        collected = []
        fetched_ids = set()
    
    to_fetch = [r for r in unique_results 
                if not r.get("has_full_text") 
                and r.get("dokumentnummer") not in fetched_ids]
    
    # Also add results that already have full text
    for r in unique_results:
        if r.get("has_full_text") and r.get("dokumentnummer") not in fetched_ids:
            collected.append(r)
            fetched_ids.add(r.get("dokumentnummer"))
    
    logger.info(f"Fetching {len(to_fetch)} documents...")
    
    for i, doc in enumerate(to_fetch):
        doc_nr = doc.get("dokumentnummer", "")
        logger.info(f"[{i+1}/{len(to_fetch)}] {doc_nr or doc.get('geschaeftszahl', '?')}")
        
        # Try SOAP first, then HTML fallback
        full_text = ""
        if FETCH_METHOD in ("soap", "both") and doc_nr:
            full_text = fetch_document_soap(doc_nr)
        
        if not full_text and doc.get("dokument_url"):
            full_text = fetch_document_html(doc["dokument_url"])
        
        doc["full_text"] = full_text
        doc["has_full_text"] = len(full_text) > 100
        doc["fetched_at"] = datetime.now().isoformat()
        collected.append(doc)
        
        # Checkpoint every 25
        if (i + 1) % 25 == 0:
            with open(CHECKPOINT_FILE, "w", encoding="utf-8") as f:
                json.dump(collected, f, ensure_ascii=False, indent=1)
            logger.info(f"  Checkpoint: {len(collected)} total")
    
    # Final save
    with open(CHECKPOINT_FILE, "w", encoding="utf-8") as f:
        json.dump(collected, f, ensure_ascii=False, indent=1)

logger.info(f"\n✅ Collection complete: {len(collected)} decisions")

## 5. Build Analysis DataFrame

In [ ]:
# --- Build clean DataFrame ---

df = pd.DataFrame(collected)

# Parse date
df["date_parsed"] = pd.to_datetime(df["entscheidungsdatum"], errors="coerce")
df["year"] = df["date_parsed"].dt.year

# Text length
df["text_length"] = df["full_text"].fillna("").str.len()

# Keyword presence flags
for kw in KEYWORDS:
    col = f"contains_{kw.lower().replace(' ', '_')}"
    df[col] = df["full_text"].fillna("").str.contains(kw, case=False, na=False)

# Drop empty texts
empty = (df["text_length"] < 100).sum()
print(f"Decisions with <100 chars text: {empty}")
df_clean = df[df["text_length"] >= 100].copy()

print(f"\nFinal dataset: {len(df_clean)} decisions")
print(f"Year range: {df_clean['year'].min()} – {df_clean['year'].max()}")
print(f"\nCourt distribution:")
print(df_clean["gericht"].value_counts().head(10))
print(f"\nSearch keyword distribution:")
# Split multi-keyword entries for counting
kw_counts = df_clean["search_keyword"].str.split("; ").explode().value_counts()
print(kw_counts)
print(f"\nText length stats:")
print(df_clean["text_length"].describe())

In [ ]:
# --- Extract referenced legal norms from text ---

def extract_norms(text):
    """Extract referenced legal norms like '§ 180 ABGB', 'Art 8 EMRK'."""
    if not text:
        return []
    pattern = r"(?:§§?|Art\.?)\s*\d+[a-z]?(?:\s*(?:bis|-)\s*\d+[a-z]?)?\s+[A-ZÄÖÜa-zäöü]+"
    return list(set(re.findall(pattern, text)))

df_clean["extracted_norms"] = df_clean["full_text"].apply(extract_norms)
df_clean["extracted_norms_str"] = df_clean["extracted_norms"].apply(lambda x: "; ".join(x))

# Most frequently cited norms
all_norms = df_clean["extracted_norms"].explode().dropna()
print("Most frequently cited norms:")
print(all_norms.value_counts().head(20))

In [ ]:
# --- KWIC: Keyword in Context ---

def extract_kwic(text, keyword, window=200):
    """Extract all occurrences of keyword with surrounding context."""
    contexts = []
    text_lower = text.lower()
    kw_lower = keyword.lower()
    start = 0
    while True:
        idx = text_lower.find(kw_lower, start)
        if idx == -1:
            break
        ctx_start = max(0, idx - window)
        ctx_end = min(len(text), idx + len(keyword) + window)
        contexts.append({
            "keyword": keyword,
            "position": idx,
            "context": text[ctx_start:ctx_end].strip(),
        })
        start = idx + 1
    return contexts

kwic_rows = []
for _, row in df_clean.iterrows():
    for kw in ["Entfremdung", "Kindeswohl"]:
        for ctx in extract_kwic(row["full_text"], kw):
            kwic_rows.append({
                "dokumentnummer": row["dokumentnummer"],
                "geschaeftszahl": row["geschaeftszahl"],
                "gericht": row["gericht"],
                "year": row["year"],
                **ctx,
            })

df_kwic = pd.DataFrame(kwic_rows)
print(f"KWIC dataset: {len(df_kwic)} keyword occurrences")
if len(df_kwic) > 0:
    print(df_kwic["keyword"].value_counts())
    print(f"\nSample context:")
    print(df_kwic.iloc[0]["context"][:300])

## 6. Export

In [ ]:
# --- Export datasets ---

# Metadata CSV (without full text, for quick inspection)
cols_meta = [c for c in df_clean.columns if c not in ["full_text", "extracted_norms"]]
df_clean[cols_meta].to_csv(OUTPUT_DIR / "decisions_metadata.csv", index=False, encoding="utf-8-sig")
print(f"✅ {OUTPUT_DIR / 'decisions_metadata.csv'}")

# Full dataset as Parquet
export_cols = [c for c in df_clean.columns if c != "extracted_norms"]  # lists don't serialize to parquet
df_clean[export_cols].to_parquet(OUTPUT_DIR / "decisions_full.parquet", index=False)
print(f"✅ {OUTPUT_DIR / 'decisions_full.parquet'}")

# KWIC contexts
if len(df_kwic) > 0:
    df_kwic.to_csv(OUTPUT_DIR / "kwic_contexts.csv", index=False, encoding="utf-8-sig")
    print(f"✅ {OUTPUT_DIR / 'kwic_contexts.csv'}")

# Individual texts (for topic modeling / BERTopic)
texts_dir = OUTPUT_DIR / "texts"
texts_dir.mkdir(exist_ok=True)
for _, row in df_clean.iterrows():
    fn = re.sub(r"[^a-zA-Z0-9_-]", "_", row.get("geschaeftszahl", "") or row.get("dokumentnummer", "unknown"))
    with open(texts_dir / f"{fn}.txt", "w", encoding="utf-8") as f:
        f.write(row["full_text"])
print(f"✅ {len(df_clean)} text files in {texts_dir}/")

print(f"\n{'='*50}")
print("All exports complete!")

## 7. Quick Visualizations

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Decisions per year
if df_clean["year"].notna().any():
    df_clean.groupby("year").size().plot(kind="bar", ax=axes[0, 0], color="steelblue")
    axes[0, 0].set_title("Decisions per Year")
    axes[0, 0].set_xlabel("")

# 2. Court distribution
df_clean["gericht"].value_counts().head(8).plot(kind="barh", ax=axes[0, 1], color="coral")
axes[0, 1].set_title("Decisions by Court")

# 3. Text length distribution
df_clean["text_length"].plot(kind="hist", bins=30, ax=axes[1, 0], color="seagreen")
axes[1, 0].set_title("Text Length Distribution")
axes[1, 0].set_xlabel("Characters")

# 4. Keyword co-occurrence
kw_cols = [c for c in df_clean.columns if c.startswith("contains_")]
kw_summary = df_clean[kw_cols].sum().sort_values(ascending=True)
kw_summary.index = [c.replace("contains_", "") for c in kw_summary.index]
kw_summary.plot(kind="barh", ax=axes[1, 1], color="mediumpurple")
axes[1, 1].set_title("Keyword Presence in Decisions")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "overview_plots.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {OUTPUT_DIR / 'overview_plots.png'}")

---

## Troubleshooting

### API returns XML instead of JSON
Add `Accept: application/json` header (already done in session setup). If it still returns XML, switch to the SOAP service or parse XML with `xml.etree.ElementTree`.

### "0 results" for a keyword
Open the equivalent URL in your browser to verify:
```
https://data.bka.gv.at/ris/api/v2.6/Judikatur?Applikation=Justiz&Suchworte=Kindeswohl&Seitengroesse=5
```
If this works in the browser but not in code, check encoding of special characters (ö, ü, etc.).

### Full text not in search results
This is normal — the search endpoint returns metadata. Use the SOAP `getDocument` method or HTML fallback as implemented in Section 3.

### Rate limiting
Default delay is 1 second. Increase `REQUEST_DELAY` if you get 429 errors. The checkpoint system lets you resume.

### Existing Python wrapper
There's a community wrapper at [github.com/PhilippTh/ris-API-wrapper](https://github.com/PhilippTh/ris-API-wrapper) built for v2.5. Install with:
```python
pip install risApiWrapper
```
Usage:
```python
from risApiWrapper.Judikatur import Justiz
results = Justiz(search_words="Kindeswohl")
for decision in results:
    print(decision["case_number"])
```
The wrapper handles pagination and error handling, but may not support all v2.6 features.

### Next steps for NLP
- Load `decisions_full.parquet` in your NLP notebook
- Use `kwic_contexts.csv` for targeted keyword analysis
- `extracted_norms_str` column is useful for legal citation network analysis
- BERTopic with `paraphrase-multilingual-MiniLM-L12-v2` works well on German legal texts